# AdaMF-MAT trên PSG — giao thức chung, tập công khai thứ hai

Bản sao `notebooks/kaggle/adamf/protocol_adamf_kaggle.ipynb`, chỉ đổi: input = dataset `huychau/mmxai-psg-bundle`, tên MMKG `PSG_*`, bật ablation (full + shuffle + drop), bỏ fallback chạy notebook VG.


In [ ]:
# ===== CONFIG (edit only here) =====
N_TRAIN_IMAGES = 6000     # train images that get "contains" edges in the MMKG (val/test images are always all included)
BUDGET_H = 2.5            # hours per training phase; epochs derived from measured speed, SAME for every variant
EPOCH_CAP = 300
LR = 1e-4
REUSE_RUN = ""            # "" = chạy cả chuỗi (~3 giờ). Hoặc đường dẫn thư mục AdaMF-MAT của lượt trước (có checkpoint/, embeddings/, benchmarks/, mmkgc/, COCO_full_meta.json) → bỏ qua build + huấn luyện, chỉ đánh giá (~15 phút)
USE_ADAPTER = True        # thêm CLIP+Adapter, CLIP+Adapter+khớp tên, CLIP+Adapter+AdaMF-MAT (mở rộng; adapter huấn luyện trên caption train, chọn epoch theo val)
RUN_ABLATIONS = True      # ablations (shuffled/dropped edges); True adds 2 more trainings (~2x BUDGET_H hours)
SEEDS = [0, 1, 2]         # gallery folds, same as the team protocol
import os, subprocess, time, re, glob, json, zipfile, traceback
import torch
N_GPU = torch.cuda.device_count()
R, OUT, SRC = "/kaggle/working/AdaMF-MAT", "/kaggle/working", "/kaggle/working/src"
if REUSE_RUN:
    R = REUSE_RUN
REPORT = f"{OUT}/report_protocol.txt"
STATE = {"ok": True, "epochs": 0, "work": None}
NAMES = {"full": "PSG_full", "shuffle": "PSG_shuf", "drop": "PSG_drop"}
VARIANTS = ["full", "shuffle", "drop"] if RUN_ABLATIONS else ["full"]
def log(*a):
    s = " ".join(str(x) for x in a); print(s, flush=True); open(REPORT, "a").write(s + "\n")
def sh(cmd, cwd=None, timeout=None):
    try:
        p = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True, timeout=timeout)
        return p.returncode, (p.stdout + p.stderr)[-3500:]
    except subprocess.TimeoutExpired as e:
        return -9, "TIMEOUT " + str(e)[-300:]
def stage(name):
    def deco(fn):
        log(f"\n######## {name} ########"); t = time.time()
        try: fn()
        except Exception: STATE["ok"] = False; log("STAGE FAILED:", traceback.format_exc()[-2500:])
        log(f"-- {name} took {time.time()-t:.0f}s"); return fn
    return deco
open(REPORT, "w").write("AdaMF-MAT on the team protocol (the mmxai_eval) report\n")
os.makedirs(SRC, exist_ok=True)
print("GPUs:", N_GPU)
print(sh("pip install -q ijson open_clip_torch")[1])

In [ ]:
@stage("1 locate the WORK dir = PSG bundle")
def _():
    log("what is attached in /kaggle/input:")
    log(sh("find /kaggle/input -maxdepth 4 | head -60")[1])
    hits = sorted(glob.glob("/kaggle/input/**/mmxai-psg-bundle/**/data/images.jsonl", recursive=True)
                  or glob.glob("/kaggle/input/**/data/images.jsonl", recursive=True))
    assert hits, "PSG bundle (huychau/mmxai-psg-bundle) chưa được gắn"
    work = os.path.dirname(os.path.dirname(hits[0]))
    log("WORK =", work); log(sh(f"ls {work} {work}/features {work}/split")[1])
    if os.path.exists(f"{work}/mmxai_eval.py"):  # use the exact module that produced the bundle
        log(sh(f"cp {work}/mmxai_eval.py {SRC}/mmxai_eval.py")[1]); log("using the bundle's mmxai_eval.py")
    assert glob.glob(f"{work}/features/clip_*.npz"), "features/clip_*.npz missing in the output"
    STATE["work"] = work

@stage("2 clone AdaMF-MAT")
def _():
    if REUSE_RUN: log("skipped (REUSE_RUN)"); return
    if not os.path.exists(R):
        log(sh(f"git clone -q https://github.com/zjukg/AdaMF-MAT {R}")[1])
    log(sh(f"ls {R}/benchmarks/MKG-W")[1])

In [ ]:
%%writefile /kaggle/working/src/mmxai_eval.py
"""mmxai_eval — module đánh giá dùng chung cho Project 3 (giao thức truy vấn & đánh giá chuẩn).

Mọi phương pháp (CLIP thuần, CLIP + Graph, phương pháp mới) đều đi qua CÙNG MỘT
giao thức: cùng file split, cùng gallery / fold theo seed, cùng top-K ứng viên CLIP, cùng hàm metric.
Một phương pháp chỉ cần cung cấp hàm  ``score_fn(cands, protocol) -> np.ndarray (Nq, K)``:
điểm cuối cùng của từng ứng viên trong top-K CLIP (điểm cao = xếp trước).

Ví dụ:
    P = Protocol.load("work_vg_data")                          # split + đặc trưng CLIP + nhóm truy vấn
    clip = lambda c, p: c["sim"]                              # baseline CLIP: điểm = cosine
    res = P.evaluate(clip, name="CLIP", split="test", seeds=[0, 1, 2])
    print(summary_table([res]))

    def clip_graph(c, p):                                     # CLIP + Graph
        sg = my_graph_score(c["q_cap"], c["cand"])            # (Nq, K) điểm đồ thị
        return fuse(c["sim"], sg, alpha=0.7)                  # α·z(cos) + (1−α)·z(s_G)
    res2 = P.evaluate(clip_graph, name="CLIP+GAT", split="test", seeds=[0, 1, 2])
    print(paired_bootstrap(res["per_seed"][0]["ranks"], res2["per_seed"][0]["ranks"]))

Thành phần:
    * chuẩn hoá văn bản không phụ thuộc thư viện ngoài (``norm_phrase``, ``ConceptLinker``)
    * metric xếp hạng: Recall@k, MRR, MedR, nDCG@10 (1 ảnh đúng) và bản nhiều ảnh liên quan
    * ``Protocol``: gallery / fold theo seed, top-K ứng viên (FAISS nếu có), đánh giá text→image và
      image→text, tách nhóm truy vấn (trực tiếp / gián tiếp)
    * ``fuse`` / ``tune_alpha``: kết hợp điểm α·z(cos) + (1−α)·z(s_G), chọn α trên validation
    * ``aggregate`` / ``summary_table`` / ``paired_bootstrap``: mean ± std theo seed, bảng Markdown, kiểm định
"""
from __future__ import annotations

import json
import os
import re
import time
from collections import Counter
from pathlib import Path

import numpy as np

__all__ = [
    "rank_metrics", "multi_relevant_metrics", "ranks_from_scores", "rerank_ranks", "zscore", "fuse",
    "tune_alpha", "aggregate", "summary_table", "paired_bootstrap", "Protocol", "ConceptLinker",
    "norm_phrase", "label_query_groups", "topk", "log", "save_json", "load_json",
]

METRICS = ["R@1", "R@5", "R@10", "MRR", "nDCG@10"]
_T0 = time.time()


def log(msg):
    print(f"[{time.time() - _T0:7.1f}s] {msg}", flush=True)


def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    def default(o):
        if isinstance(o, np.integer):
            return int(o)
        if isinstance(o, np.floating):
            return float(o)
        if isinstance(o, np.ndarray):
            return o.tolist()
        raise TypeError(type(o))

    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=1, ensure_ascii=False, default=default)


def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


# =============================================================================== text normalisation
STOPWORDS = set("""
a an the and or but if then than so of at by for with about against between into through during before
after above below to from up down in out on off over under again further once here there when where why how
all any both each few more most other some such no nor not only own same too very s t can will just don
should now is are was were be been being am have has had having do does did doing would could ought i me
my myself we our ours ourselves you your yours yourself yourselves he him his himself she her hers herself
it its itself they them their theirs themselves what which who whom this that these those while as until
one two three four five six seven eight nine ten several many lot lots something someone somebody thing
things another next near around along front top side back also very image picture photo view shot
showing shows show close closeup
""".split())
_TOKEN_RE = re.compile(r"[a-z]+")
_IRREG = {"men": "man", "women": "woman", "children": "child", "people": "people", "feet": "foot",
          "teeth": "tooth", "mice": "mouse", "geese": "goose", "knives": "knife", "leaves": "leaf",
          "shelves": "shelf", "wolves": "wolf", "glasses": "glasses", "jeans": "jeans", "pants": "pants",
          "shorts": "shorts", "scissors": "scissors", "bus": "bus", "grass": "grass", "dress": "dress",
          "skis": "ski", "fries": "fries", "sunglasses": "sunglasses", "clothes": "clothes", "tennis": "tennis",
          "news": "news", "lens": "lens", "gas": "gas", "canvas": "canvas", "atlas": "atlas", "chess": "chess"}


def rule_lemma(w):
    """Lemmatiser luật đơn giản (không cần NLTK): 'dogs'->'dog', 'buses'->'bus', 'riding'->'rid' (chấp nhận)."""
    if w in _IRREG or w.endswith(("us", "is", "ss")):
        return _IRREG.get(w, w)
    if len(w) > 4 and w.endswith("ies"):
        return w[:-3] + "y"
    if len(w) > 4 and w.endswith("es") and w[-3] in "sxzh":
        return w[:-2]
    if len(w) > 3 and w.endswith("s") and not w.endswith("ss"):
        return w[:-1]
    return w


def tokenize(text):
    return _TOKEN_RE.findall(text.lower())


def norm_phrase(text, max_words=3):
    """'Tennis Rackets' -> 'tennis racket'. Trả về None nếu rỗng hoặc quá dài."""
    toks = [rule_lemma(t) for t in tokenize(str(text).replace("_", " "))]
    toks = [t for t in toks if t and t not in STOPWORDS] or toks
    if not toks or len(toks) > max_words:
        return None
    return " ".join(toks)


class ConceptLinker:
    """Khớp tham lam n-gram (dài nhất trước) của truy vấn với từ vựng khái niệm (tên đối tượng VG)."""

    def __init__(self, vocab, max_ngram=3):
        self.vocab = set(vocab)
        self.max_ngram = max_ngram

    def link(self, text):
        seq = [rule_lemma(t) for t in tokenize(text)]
        out, i = [], 0
        while i < len(seq):
            hit = None
            for n in range(min(self.max_ngram, len(seq) - i), 0, -1):
                gram = seq[i:i + n]
                if gram[0] in STOPWORDS or gram[-1] in STOPWORDS:
                    continue
                name = " ".join(gram)
                if name in self.vocab:
                    hit = (name, n)
                    break
            if hit is None:
                i += 1
                continue
            if hit[0] not in out:
                out.append(hit[0])
            i += hit[1]
        return out


# Màu sắc / thuộc tính / danh từ chung chung xuất hiện như "đối tượng" trong VG nhưng không phải thực thể thị giác —
# bỏ khỏi việc GÁN NHÓM truy vấn (đồ thị vẫn có thể dùng chúng).
GENERIC_CONCEPTS = set("""
white black blue green brown yellow orange red pink gray grey purple gold silver tan dark light color colour
group bunch area piece background foreground day night air body row part side top front middle number couple pair
lot variety kind type set picture image photo scene view stand spot line edge corner shape pattern design
""".split())


def label_query_groups(images, captions, vocab, generic=GENERIC_CONCEPTS):
    """Nhóm truy vấn theo scene graph của ảnh đúng (chỉ để BÁO CÁO, mô hình không dùng).

    direct     : mọi khái niệm liên kết được từ caption đều có trong objects của ảnh đúng
    indirect   : có ít nhất 1 khái niệm KHÔNG có trong ảnh (chỉ nối được qua quan hệ / đồng nghĩa)
    no_concept : caption không liên kết được khái niệm nào trong từ vựng
    """
    linker = ConceptLinker(vocab)
    groups, concepts = [], []
    for c in captions:
        cs = [x for x in linker.link(c["text"]) if x not in generic]
        objs = set(images[c["img"]]["objects"])
        concepts.append(cs)
        if not cs:
            groups.append("no_concept")
        elif all(x in objs for x in cs):
            groups.append("direct")
        else:
            groups.append("indirect")
    return np.array(groups), concepts


# =============================================================================== metrics
def rank_metrics(ranks, ks=(1, 5, 10)):
    """Metric khi mỗi truy vấn có đúng 1 ảnh đúng. ``ranks``: hạng (1-based) của ảnh đúng."""
    r = np.asarray(ranks, dtype=np.float64)
    m = {f"R@{k}": float((r <= k).mean() * 100) for k in ks}
    m["MRR"] = float((1.0 / r).mean() * 100)
    m["nDCG@10"] = float(np.where(r <= 10, 1.0 / np.log2(r + 1), 0.0).mean() * 100)
    m["MedR"] = float(np.median(r))
    m["n"] = int(len(r))
    return m


def multi_relevant_metrics(order, relevant, ks=(1, 5, 10)):
    """Metric khi một truy vấn có NHIỀU ảnh liên quan (bộ ≥20 truy vấn).

    ``order``: danh sách id ảnh đã xếp hạng; ``relevant``: tập id ảnh liên quan.
    """
    hits = np.array([i in relevant for i in order], dtype=float)
    first = int(np.argmax(hits)) + 1 if hits.any() else len(order) + 1
    m = {f"R@{k}": float(hits[:k].any() * 100) for k in ks}
    m["MRR"] = 100.0 / first
    m["P@10"] = float(hits[:10].mean() * 100)
    disc = 1.0 / np.log2(np.arange(2, 12))
    ideal = disc[: min(len(relevant), 10)].sum()
    m["nDCG@10"] = float((hits[:10] * disc[: len(hits[:10])]).sum() / ideal * 100) if ideal else 0.0
    return m


def ranks_from_scores(scores, gt_col):
    """Hạng (1-based) của cột gt_col trong từng hàng điểm. Hoà điểm: xếp ảnh đúng trước (đếm 'lớn hơn hẳn')."""
    gt = scores[np.arange(len(scores)), gt_col][:, None]
    return (scores > gt).sum(1) + 1


def rerank_ranks(cands, scores):
    """Hạng ảnh đúng sau khi xếp lại top-K bằng ``scores`` (Nq, K); ảnh đúng ngoài top-K giữ hạng CLIP."""
    pos = cands["pos"]
    order = np.argsort(-scores, axis=1, kind="stable")
    new_pos = np.argmax(order == pos[:, None], axis=1)
    return np.where(pos >= 0, new_pos + 1, cands["gt_rank"])


def zscore(x):
    mu, sd = x.mean(1, keepdims=True), x.std(1, keepdims=True)
    return np.where(sd > 1e-8, (x - mu) / np.maximum(sd, 1e-8), 0.0)


def fuse(sim, graph_score, alpha):
    """score = α·z(cos) + (1−α)·z(s_G), chuẩn hoá z-score trong top-K của từng truy vấn."""
    return alpha * zscore(np.asarray(sim, np.float64)) + (1 - alpha) * zscore(np.asarray(graph_score, np.float64))


def tune_alpha(cands, graph_score, step=0.05, key="MRR"):
    """Quét α ∈ [0,1] trên một split (PHẢI là validation). Trả về (α tốt nhất, đường cong)."""
    curve = []
    for a in np.round(np.arange(0, 1 + 1e-9, step), 4):
        m = rank_metrics(rerank_ranks(cands, fuse(cands["sim"], graph_score, a)))
        curve.append(dict(alpha=float(a), **m))
    best = max(curve, key=lambda d: (d[key], d["alpha"]))
    return best["alpha"], curve


def aggregate(list_of_metrics, keys=None):
    keys = keys or [k for k in list_of_metrics[0] if isinstance(list_of_metrics[0][k], (int, float)) and k != "n"]
    out = {}
    for k in keys:
        v = np.array([m[k] for m in list_of_metrics], dtype=float)
        out[k] = dict(mean=float(v.mean()), std=float(v.std(ddof=0)), values=v.tolist())
    out["n"] = int(np.mean([m.get("n", 0) for m in list_of_metrics]))
    return out


def _cell(v, digits=2):
    if isinstance(v, dict):
        return f"{v['mean']:.{digits}f} ± {v['std']:.{digits}f}"
    return f"{v:.{digits}f}" if isinstance(v, float) else str(v)


def summary_table(results, metrics=METRICS, field="mean_std", title=None):
    """Bảng Markdown: một dòng / kết quả của ``Protocol.evaluate``."""
    lines = ([f"**{title}**", ""] if title else []) + [
        "| phương pháp | split | n truy vấn | " + " | ".join(metrics) + " |",
        "|---|---|---|" + "---|" * len(metrics)]
    for r in results:
        src = r[field]
        lines.append(f"| {r['name']} | {r['split']} | {src.get('n', '')} | " + " | ".join(_cell(src[m]) for m in metrics) + " |")
    return "\n".join(lines)


def paired_bootstrap(ranks_a, ranks_b, n_boot=2000, seed=0):
    """So sánh 2 phương pháp trên CÙNG các truy vấn (ghép cặp) bằng bootstrap trên hiệu MRR và R@1.

    Trả về hiệu (b − a), khoảng tin cậy 95 % và p-value hai phía ≈ 2·min(P(Δ≤0), P(Δ≥0)).
    """
    a, b = np.asarray(ranks_a, float), np.asarray(ranks_b, float)
    assert a.shape == b.shape, "hai phương pháp phải chạy trên cùng danh sách truy vấn"
    rng = np.random.RandomState(seed)
    out = {}
    for name, fa in (("MRR", lambda r: 100 / r), ("R@1", lambda r: 100.0 * (r <= 1)), ("R@10", lambda r: 100.0 * (r <= 10))):
        d = fa(b) - fa(a)
        idx = rng.randint(0, len(d), size=(n_boot, len(d)))
        boots = d[idx].mean(1)
        p = 2 * min((boots <= 0).mean(), (boots >= 0).mean())
        out[name] = dict(delta=float(d.mean()), ci95=[float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))],
                         p_value=float(min(1.0, p)))
    return out


# =============================================================================== similarity / top-K
def _device():
    try:
        import torch
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    except Exception:  # noqa: BLE001
        return None


def topk(q, gal, K, use_faiss=True):
    """Top-K theo tích vô hướng (cosine nếu đã chuẩn hoá). Trả về (sim (Nq,K), idx (Nq,K), sims đầy đủ hoặc None)."""
    q = np.ascontiguousarray(q, dtype=np.float32)
    gal = np.ascontiguousarray(gal, dtype=np.float32)
    K = min(K, len(gal))
    if use_faiss:
        try:
            import faiss
            index = faiss.IndexFlatIP(gal.shape[1])
            index.add(gal)
            s, i = index.search(q, K)
            return s, i, index
        except Exception:  # noqa: BLE001
            pass
    sims = q @ gal.T
    i = np.argpartition(-sims, K - 1, axis=1)[:, :K]
    s = np.take_along_axis(sims, i, 1)
    o = np.argsort(-s, axis=1, kind="stable")
    return np.take_along_axis(s, o, 1), np.take_along_axis(i, o, 1), None


def full_sims(q, gal, batch=4096):
    """Ma trận cosine đầy đủ (Nq, Ng), dùng GPU nếu có."""
    dev = _device()
    q = np.asarray(q, np.float32)
    gal = np.asarray(gal, np.float32)
    if dev is not None and dev.type == "cuda":
        import torch
        g = torch.tensor(gal, device=dev)
        out = [(torch.tensor(q[b:b + batch], device=dev) @ g.T).cpu().numpy() for b in range(0, len(q), batch)]
        return np.concatenate(out)
    return q @ gal.T


# =============================================================================== protocol
class Protocol:
    """Giao thức đánh giá chung.

    Dữ liệu (do notebook giao thức đánh giá tạo ra trong ``work_dir``):
        data/images.jsonl   : mỗi dòng 1 ảnh {idx, id, vg_id, coco_id, url, split, objects, relations}
        data/captions.jsonl : mỗi dòng 1 caption {idx, img, text, group}
        split/split.json    : {"train": [...coco_id], "val": [...], "test": [...], "meta": {...}}
        features/clip_<model>.npz : img (Ni,D) float16, cap (Nc,D) float16, cap_img (Nc,), ok (Ni,) bool

    Seed:  CLIP đóng băng là tất định, nên ``seed`` điều khiển (i) fold gallery ``gallery_size`` ảnh lấy
    mẫu từ pool test/val, (ii) thứ tự gallery train. Mô hình học (GNN) dùng cùng seed để khởi tạo.
    ``seed=None`` → dùng toàn bộ pool (kết quả tất định, dùng làm số chính).
    """

    def __init__(self, images, captions, split, feats, gallery_size=1000, K=100, seed_data=0):
        self.images, self.captions, self.split = images, captions, split
        self.img = np.asarray(feats["img"], np.float32)
        self.cap = np.asarray(feats["cap"], np.float32)
        self.cap_img = np.asarray(feats["cap_img"])
        self.ok = np.asarray(feats["ok"], bool)
        self.G, self.K, self.seed_data = gallery_size, K, seed_data
        self.caps_of = {}
        for ci, ii in enumerate(self.cap_img):
            self.caps_of.setdefault(int(ii), []).append(ci)
        self.groups = np.array([c.get("group", "all") for c in captions])
        self._cache = {}

    # ---- loading
    @classmethod
    def load(cls, work_dir, feature_file=None, **kw):
        work = Path(work_dir)
        images = [json.loads(l) for l in open(work / "data" / "images.jsonl", encoding="utf-8")]
        captions = [json.loads(l) for l in open(work / "data" / "captions.jsonl", encoding="utf-8")]
        split = load_json(work / "split" / "split.json")
        feature_file = feature_file or sorted((work / "features").glob("clip_*.npz"))[0]
        feats = np.load(feature_file, allow_pickle=False)
        meta = split.get("meta", {})
        kw.setdefault("gallery_size", meta.get("gallery_size", 1000))
        kw.setdefault("K", meta.get("top_k", 100))
        kw.setdefault("seed_data", meta.get("seed_data", 0))
        log(f"Protocol: {len(images)} ảnh, {len(captions)} caption, features {Path(feature_file).name} "
            f"{feats['img'].shape}, gallery={kw['gallery_size']}, K={kw['K']}")
        return cls(images, captions, split, feats, **kw)

    # ---- galleries
    def pool(self, split):
        return [im["idx"] for im in self.images if im["split"] == split and self.ok[im["idx"]]]

    def gallery(self, split, seed=None):
        pool = self.pool(split)
        if seed is None or len(pool) <= self.G:
            return sorted(pool)
        rng = np.random.RandomState(seed)
        return sorted(rng.choice(pool, self.G, replace=False).tolist())

    def train_galleries(self, seed=None):
        """Train chia thành các gallery ``gallery_size`` ảnh (giống test) để lấy hard negative."""
        pool = self.pool("train")
        rng = np.random.RandomState(self.seed_data if seed is None else seed)
        rng.shuffle(pool)
        return [pool[i:i + self.G] for i in range(0, len(pool), self.G)]

    # ---- candidates
    def candidates(self, split="test", seed=None, galleries=None, ensure_positive=False, captions_per_image=None):
        """Top-K ứng viên CLIP cho mọi caption của ảnh trong gallery (định dạng chung cho mọi phương pháp).

        Trả về dict: q_cap (Nq,) idx caption, q_img (Nq,) idx ảnh đúng, cand (Nq,K) idx ảnh ứng viên,
        sim (Nq,K) cosine, pos (Nq,) cột của ảnh đúng trong cand (−1 nếu ngoài top-K),
        gt_rank (Nq,) hạng CLIP của ảnh đúng trong toàn gallery, gallery (list idx), group (Nq,).
        """
        key = (split, seed, galleries is None, ensure_positive, captions_per_image)
        if key in self._cache:
            return self._cache[key]
        if galleries is None:
            galleries = self.train_galleries(seed) if split == "train" else [self.gallery(split, seed)]
        rng = np.random.RandomState(0 if seed is None else seed)
        out = dict(q_cap=[], q_img=[], cand=[], sim=[], pos=[], gt_rank=[])
        for gal in galleries:
            gal = list(gal)
            col = {g: j for j, g in enumerate(gal)}
            q_cap = []
            for g in gal:
                cs = self.caps_of.get(g, [])
                if captions_per_image and len(cs) > captions_per_image:
                    cs = sorted(rng.choice(cs, captions_per_image, replace=False).tolist())
                q_cap += cs
            q_cap = np.array(q_cap, dtype=np.int64)
            q_img = self.cap_img[q_cap].astype(np.int64)
            sims = full_sims(self.cap[q_cap], self.img[gal])
            gt_col = np.array([col[int(i)] for i in q_img])
            gt_rank = ranks_from_scores(sims, gt_col)
            K = min(self.K, len(gal))
            idx = np.argsort(-sims, axis=1, kind="stable")[:, :K]
            sim = np.take_along_axis(sims, idx, 1).astype(np.float32)
            cand = np.array(gal)[idx]
            hit = cand == q_img[:, None]
            pos = np.where(hit.any(1), hit.argmax(1), -1)
            if ensure_positive:  # train: ảnh đúng luôn nằm trong ứng viên (thay ứng viên cuối)
                miss = pos < 0
                cand[miss, -1] = q_img[miss]
                sim[miss, -1] = sims[miss, gt_col[miss]]
                pos[miss] = K - 1
            for k, v in zip(out, (q_cap, q_img, cand, sim, pos, gt_rank)):
                out[k].append(v)
        res = {k: np.concatenate(v) for k, v in out.items()}
        res["gallery"] = [int(x) for g in galleries for x in g]
        res["gallery_size"] = len(galleries[0])
        res["group"] = self.groups[res["q_cap"]]
        res["split"], res["seed"] = split, seed
        self._cache[key] = res
        return res

    # ---- evaluation text -> image
    def evaluate(self, score_fn, name, split="test", seeds=(0, 1, 2), full=True, by_group=True, verbose=True):
        """Chạy ``score_fn`` trên từng fold theo seed (+ toàn bộ pool nếu ``full``) và gộp mean ± std."""
        per_seed, groups = [], {}
        for seed in list(seeds) + ([None] if full else []):
            c = self.candidates(split, seed)
            scores = np.asarray(score_fn(c, self), dtype=np.float64)
            assert scores.shape == c["sim"].shape, f"score_fn phải trả về (Nq,K)={c['sim'].shape}, nhận {scores.shape}"
            ranks = rerank_ranks(c, scores)
            m = rank_metrics(ranks)
            rec = dict(seed=seed, gallery_size=c["gallery_size"], ranks=ranks.tolist(), **m)
            if by_group:
                rec["by_group"] = {g: rank_metrics(ranks[c["group"] == g]) for g in np.unique(c["group"])}
            per_seed.append(rec)
            if verbose:
                tag = "full" if seed is None else f"seed {seed}"
                log(f"{name:>24s} | {split} {tag:>6s} | gallery {c['gallery_size']:5d} | " +
                    " ".join(f"{k}={m[k]:.2f}" for k in METRICS) + f" MedR={m['MedR']:.0f}")
        seeded = [r for r in per_seed if r["seed"] is not None]
        res = dict(name=name, split=split, seeds=list(seeds), gallery_size=self.G, per_seed=per_seed,
                   mean_std=aggregate(seeded) if seeded else None,
                   full=next((r for r in per_seed if r["seed"] is None), None))
        if by_group and seeded:
            for g in np.unique(self.groups):
                vals = [r["by_group"][g] for r in seeded if g in r["by_group"]]
                if vals:
                    groups[g] = aggregate(vals)
            res["by_group"] = groups
        return res

    # ---- evaluation image -> text (nhánh B)
    def evaluate_i2t(self, name="CLIP", split="test", seeds=(0, 1, 2), full=True, score_fn=None, verbose=True):
        """Truy vấn = ảnh, gallery = mọi caption của ảnh trong gallery; đúng nếu lấy được 1 trong 5 caption.

        ``score_fn(sims, protocol, gallery, cap_idx) -> (Ni, Nc)`` để cắm phương pháp khác (mặc định: cosine).
        """
        per_seed = []
        for seed in list(seeds) + ([None] if full else []):
            gal = self.gallery(split, seed)
            cap_idx = np.array([ci for g in gal for ci in self.caps_of.get(g, [])])
            sims = full_sims(self.img[gal], self.cap[cap_idx])
            if score_fn is not None:
                sims = np.asarray(score_fn(sims, self, gal, cap_idx), np.float64)
            owner = self.cap_img[cap_idx]
            ranks = []
            for r, g in enumerate(gal):
                mine = np.where(owner == g)[0]
                ranks.append(int(ranks_from_scores(sims[r][None, :].repeat(len(mine), 0), mine).min()))
            m = rank_metrics(ranks)
            per_seed.append(dict(seed=seed, gallery_size=len(gal), ranks=ranks, **m))
            if verbose:
                tag = "full" if seed is None else f"seed {seed}"
                log(f"{name + ' (i2t)':>24s} | {split} {tag:>6s} | gallery {len(gal):5d} | " +
                    " ".join(f"{k}={m[k]:.2f}" for k in METRICS) + f" MedR={m['MedR']:.0f}")
        seeded = [r for r in per_seed if r["seed"] is not None]
        return dict(name=name + " (image→text)", split=split, seeds=list(seeds), per_seed=per_seed,
                    mean_std=aggregate(seeded) if seeded else None,
                    full=next((r for r in per_seed if r["seed"] is None), None))

    # ---- helpers for other methods
    def image_objects(self, idx):
        return self.images[int(idx)]["objects"]

    def caption_text(self, idx):
        return self.captions[int(idx)]["text"]

    def object_vocab(self, split="train", min_images=5, max_size=2000):
        """Từ vựng tên đối tượng đếm trên ảnh TRAIN (không rò rỉ từ val/test)."""
        cnt = Counter()
        for im in self.images:
            if im["split"] == split:
                cnt.update(set(im["objects"]))
        return [n for n, c in cnt.most_common(max_size) if c >= min_images]


In [ ]:
%%writefile /kaggle/working/src/features.py
"""Frozen CLIP features (open_clip) + a fake encoder for CPU smoke tests.

Real:  Encoder("clip", img_dirs=[...])   Fake: Encoder("fake", images=images)
Both give .images(ids) -> (kept_ids, [n,d] L2-normed) and .texts(list) -> [m,d] L2-normed.
"""
import hashlib
import os

import numpy as np


def _unit(x):
    return x / (np.linalg.norm(x, axis=-1, keepdims=True) + 1e-9)


class Encoder:
    def __init__(self, mode="clip", img_dirs=(), images=None, model="ViT-B-32", pretrained="laion2b_s34b_b79k",
                 device=None, batch=256, workers=4):
        self.mode, self.img_dirs, self.images_meta = mode, list(img_dirs), images
        if mode == "clip":
            import open_clip
            import torch
            self.torch = torch
            self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
            self.model, _, self.pre = open_clip.create_model_and_transforms(model, pretrained=pretrained)
            self.model = self.model.to(self.device).eval()
            if self.device == "cuda":
                self.model = self.model.half()
            self.tok = open_clip.get_tokenizer(model)
            self.batch, self.workers = batch, workers

    # ---- fake: deterministic word vectors, images = mean of their object words + noise
    def _w(self, word, d=64):
        h = int(hashlib.md5(word.encode()).hexdigest()[:8], 16)
        return np.random.RandomState(h).randn(d)

    def _fake_text(self, t):
        return np.mean([self._w(w) for w in t.split()], axis=0)

    def texts(self, items):
        if self.mode == "fake":
            return _unit(np.stack([self._fake_text(t) for t in items]).astype("float32"))
        out = []
        with self.torch.no_grad():
            for i in range(0, len(items), 512):
                tk = self.tok(items[i:i + 512]).to(self.device)
                out.append(self.model.encode_text(tk).float().cpu().numpy())
        return _unit(np.concatenate(out))

    def _find(self, iid):
        for d in self.img_dirs:
            p = os.path.join(d, f"{iid}.jpg")
            if os.path.exists(p):
                return p
        return None

    def images(self, ids):
        if self.mode == "fake":
            rs = np.random.RandomState(0)
            f = [np.mean([self._w(w) for w in self.images_meta[i]["objs"]] or [np.zeros(64)], axis=0)
                 + 0.5 * rs.randn(64) for i in ids]
            return list(ids), _unit(np.stack(f).astype("float32"))
        from PIL import Image
        torch = self.torch
        paths = [(i, self._find(i)) for i in ids]
        paths = [(i, p) for i, p in paths if p]
        print(f"images found: {len(paths)}/{len(ids)}")

        class DS(torch.utils.data.Dataset):
            def __len__(s):
                return len(paths)

            def __getitem__(s, k):
                return self.pre(Image.open(paths[k][1]).convert("RGB"))

        dl = torch.utils.data.DataLoader(DS(), batch_size=self.batch, num_workers=self.workers)
        out = []
        with torch.no_grad():
            for x in dl:
                x = x.to(self.device)
                x = x.half() if self.device == "cuda" else x
                out.append(self.model.encode_image(x).float().cpu().numpy())
        return [i for i, _ in paths], _unit(np.concatenate(out))


In [ ]:
%%writefile /kaggle/working/src/adamf_protocol.py
"""AdaMF-MAT as a score_fn for the team's shared protocol (the mmxai_eval.Protocol).   [cấp độ 2]

Pipeline (same split, same 1K-gallery folds, same CLIP top-K as every other method):
  caption -> concepts (the ConceptLinker, already in captions.jsonl["concepts"])
  graph_score(caption, candidate image) = sum over the caption's concepts c of AdaMF-MAT plausibility of (image, contains, c)
  final score = alpha*z(cosine) + (1-alpha)*z(graph_score), alpha chosen on VAL  (mmxai_eval.fuse / tune_alpha)

MMKG that AdaMF-MAT is trained on (built from the Protocol, no leakage):
  entities  : concepts = P.object_vocab("train")  +  images (sampled train images + ALL val/test images)
  relations : "contains" + top predicates
  train2id  : (image, contains, concept) for every included image (object list = input, not label)
              + concept->concept relation triples seen in >= min_triple TRAIN images only
  valid/test: concept relation triples seen only in val/test images (new facts), used by the repo's own KGC test
  embeddings: visual = CLIP image feature (concept = mean over TRAIN images containing it)
              textual = CLIP text feature of the concept name (image = mean of its concepts' text features)
  variants  : full | shuffle (tails permuted) | drop (no concept relations) for the relation-structure ablation.

Usage (Kaggle, after the notebook produced WORK=/kaggle/working/vg_data and the repo is cloned):
  python adamf_protocol.py build --work WORK --repo REPO --name COCO_full --variant full
  (train with the repo: python run_adamf_mat.py -dataset=COCO_full ... -save=./checkpoint/COCO_full-full)
  python adamf_protocol.py eval  --work WORK --repo REPO --name COCO_full --ckpt checkpoint/COCO_full-full --out results.json
"""
import argparse
import json
import os
import shutil
import subprocess
import sys

import numpy as np

try:
    import mmxai_eval  # noqa: F401  (Kaggle notebook writes it next to this file)
except ImportError:  # in the team repo the shared module lives in ../../thang (read only, never edit it from here)
    sys.path.append(os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "..", "thang"))

SUFFIX = {"full": "", "shuffle": "_shuf", "drop": "_drop"}


def _unit(x):
    return x / (np.linalg.norm(x, axis=-1, keepdims=True) + 1e-9)


# ----------------------------------------------------------------------------------------- build the MMKG
def build_mmkg(P, repo, name, text_encode, variant="full", n_train_images=6000, min_triple=2, top_pred=30,
               seed=0, run_nn=True):
    """Write benchmarks/<name>/*.txt, embeddings/<name>-{visual,textual}.pth and <repo>/<name>_meta.json."""
    import torch
    vocab = P.object_vocab("train")
    cidx = {c: k for k, c in enumerate(vocab)}
    C = len(vocab)

    pool_tr, pool_va, pool_te = P.pool("train"), P.pool("val"), P.pool("test")
    sub = sorted(np.random.RandomState(seed).choice(pool_tr, min(n_train_images, len(pool_tr)), replace=False).tolist())
    kg_imgs = sub + sorted(pool_va) + sorted(pool_te)  # entity order after the concepts
    row_of = {ii: C + k for k, ii in enumerate(kg_imgs)}

    from collections import Counter

    def rel_counts(idxs):
        cnt = Counter()
        for ii in idxs:
            seen = {(s, p, o) for s, p, o in P.images[ii]["relations"] if s in cidx and o in cidx and s != o}
            cnt.update(seen)
        return cnt

    tr = rel_counts(pool_tr)  # ALL train images feed the concept relations (cheap), only `sub` get contains edges
    preds = [p for p, _ in Counter(p for (_, p, _), c in tr.items() for _ in range(c)).most_common(top_pred)]
    pidx = {p: k + 1 for k, p in enumerate(preds)}
    train_rel = sorted(t for t, c in tr.items() if c >= min_triple and t[1] in pidx)
    held = sorted(t for t in rel_counts(pool_va + pool_te) if t not in tr and t[1] in pidx)
    np.random.RandomState(seed).shuffle(held)
    valid_rel, test_rel = held[: len(held) // 2], held[len(held) // 2:]
    if not test_rel:
        test_rel = train_rel[:1]

    contains = [(row_of[ii], cidx[o], 0) for ii in kg_imgs for o in P.images[ii]["objects"] if o in cidx]
    rows = [(cidx[s], cidx[o], pidx[p]) for s, p, o in train_rel]
    if variant == "drop":
        rows = []
    elif variant == "shuffle":
        tails = np.random.RandomState(seed).permutation([t for _, t, _ in rows])
        rows = [(h, int(t2), r) for (h, _, r), t2 in zip(rows, tails)]

    vname = name
    bdir = os.path.join(repo, "benchmarks", vname)
    os.makedirs(bdir, exist_ok=True)
    os.makedirs(os.path.join(repo, "embeddings"), exist_ok=True)

    def write(fname, rr):
        with open(os.path.join(bdir, fname), "w") as f:
            f.write(f"{len(rr)}\n")
            for h, t, r in rr:
                f.write(f"{h} {t} {r}\n")

    write("train2id.txt", contains + rows)
    write("valid2id.txt", [(cidx[s], cidx[o], pidx[p]) for s, p, o in valid_rel])
    write("test2id.txt", [(cidx[s], cidx[o], pidx[p]) for s, p, o in test_rel])
    ents = [f"c:{c.replace(' ', '_')}" for c in vocab] + [f"img:{ii}" for ii in kg_imgs]
    rels = ["contains"] + [p.replace(" ", "_") for p in preds]
    for fname, names in (("entity2id.txt", ents), ("relation2id.txt", rels)):
        with open(os.path.join(bdir, fname), "w") as f:
            f.write(f"{len(names)}\n")
            for k, n in enumerate(names):
                f.write(f"{n.replace(' ', '_')} {k}\n")

    # embeddings
    T = _unit(np.asarray(text_encode([f"a photo of a {c}" for c in vocab]), np.float32))
    d = T.shape[1]
    assert d == P.img.shape[1], f"text dim {d} != image dim {P.img.shape[1]}"
    vis = np.zeros((len(ents), d), "float32")
    txt = np.zeros((len(ents), d), "float32")
    txt[:C] = T
    sums, ns = np.zeros((C, d), "float32"), np.zeros(C)
    for ii in pool_tr:
        for o in P.images[ii]["objects"]:
            if o in cidx:
                sums[cidx[o]] += P.img[ii]
                ns[cidx[o]] += 1
    vis[:C] = sums / np.maximum(ns, 1)[:, None]
    for ii in kg_imgs:
        k = row_of[ii]
        vis[k] = P.img[ii]
        cs = [cidx[o] for o in P.images[ii]["objects"] if o in cidx]
        if cs:
            txt[k] = _unit(T[cs].mean(0))
    emb = os.path.join(repo, "embeddings")
    torch.save(torch.tensor(vis), f"{emb}/{vname}-visual.pth")
    torch.save(torch.tensor(txt), f"{emb}/{vname}-textual.pth")
    meta = dict(name=vname, variant=variant, vocab=vocab, preds=preds, n_concepts=C, kg_images=kg_imgs, dim=int(d))
    json.dump(meta, open(os.path.join(repo, f"{vname}_meta.json"), "w"))
    print(f"[{vname}] {len(ents)} entities ({C} concepts, {len(kg_imgs)} images), {len(rels)} relations, "
          f"{len(contains) + len(rows)} train triples ({len(contains)} contains, {len(rows)} concept relations), "
          f"heldout valid/test {len(valid_rel)}/{len(test_rel)}")
    if run_nn:  # the repo's own script writes type_constrain.txt, 1-1.txt ... test2id_all.txt
        shutil.copy(os.path.join(repo, "benchmarks", "MKG-W", "n-n.py"), os.path.join(bdir, "n-n.py"))
        subprocess.run(["python", "n-n.py"], cwd=bdir, check=True)
    return meta


# ----------------------------------------------------------------------------------------- the scorer
class AdamfScorer:
    """graph_score(c, P) -> (Nq, K): AdaMF-MAT plausibility of (candidate image, contains, concept) summed over the caption's concepts."""

    def __init__(self, P, repo, name, ckpt, dim=200, margin=12.0, device=None):
        import torch
        sys.path.insert(0, repo)
        from mmkgc.module.model import AdvMixRotatE  # repo code (license not checked: do not redistribute)
        self.torch = torch
        self.dev = device or ("cuda" if torch.cuda.is_available() else "cpu")
        meta = json.load(open(os.path.join(repo, f"{name}_meta.json")))
        self.cidx = {c: k for k, c in enumerate(meta["vocab"])}
        self.C = meta["n_concepts"]
        self.row = np.full(len(P.images), -1, np.int64)  # protocol image idx -> entity id
        for k, ii in enumerate(meta["kg_images"]):
            self.row[ii] = self.C + k
        img_emb = torch.load(os.path.join(repo, "embeddings", f"{name}-visual.pth"))
        txt_emb = torch.load(os.path.join(repo, "embeddings", f"{name}-textual.pth"))
        ent_tot = int(open(os.path.join(repo, "benchmarks", name, "entity2id.txt")).readline())
        rel_tot = int(open(os.path.join(repo, "benchmarks", name, "relation2id.txt")).readline())
        self.model = AdvMixRotatE(ent_tot=ent_tot, rel_tot=rel_tot, dim=dim, margin=margin, epsilon=2.0,
                                  img_emb=img_emb, text_emb=txt_emb)
        if ckpt:  # tests may pass None to keep random weights
            self.model.load_state_dict(torch.load(os.path.join(repo, ckpt), map_location="cpu"))
        self.model = self.model.to(self.dev).eval()
        self.imgs_ent = torch.arange(self.C, ent_tot, device=self.dev)
        self._vec = {}

    def _concept_vec(self, c):
        """plausibility of (img, contains, c) for EVERY image entity, cached per concept."""
        if c not in self._vec:
            t, h = self.torch.full_like(self.imgs_ent, self.cidx[c]), self.imgs_ent
            with self.torch.no_grad():
                s = self.model({"batch_h": h, "batch_t": t, "batch_r": self.torch.zeros_like(h), "mode": "normal"})
            self._vec[c] = s.float().cpu().numpy()
        return self._vec[c]

    def __call__(self, c, P):
        out = np.zeros(c["sim"].shape, np.float64)
        rows = self.row[c["cand"]] - self.C  # (Nq,K) position in the image-entity block, negative = not in the MMKG
        for r, q in enumerate(c["q_cap"]):
            cs = [x for x in P.captions[int(q)].get("concepts", []) if x in self.cidx]
            for x in cs:
                v = self._concept_vec(x)
                ok = rows[r] >= 0
                out[r, ok] += v[rows[r][ok]]
        return out


def presence_score(c, P):
    """Control with NO relational knowledge: share of the caption's concepts in the candidate's object list (the lexical rerank)."""
    obj = [set(im["objects"]) for im in P.images]
    out = np.zeros(c["sim"].shape, np.float64)
    for r, q in enumerate(c["q_cap"]):
        cs = set(P.captions[int(q)].get("concepts", []))
        if cs:
            out[r] = [len(cs & obj[ii]) / len(cs) for ii in c["cand"][r]]
    return out


# ----------------------------------------------------------------------------------------- evaluate on the shared protocol
def evaluate_all(P, scorer, seeds=(0, 1, 2), name="CLIP+AdaMF-MAT", adapter=None):
    """adapter (tùy chọn, `adapter.Adapter` đã huấn luyện): thêm CLIP+Adapter, +khớp tên, +AdaMF-MAT. Mốc đối chứng đúng cho AdaMF-MAT
    khi có adapter là CLIP+Adapter (không phải CLIP thô), nên bootstrap của các phương pháp có adapter lấy mốc đó."""
    from mmxai_eval import fuse, paired_bootstrap, summary_table, tune_alpha
    base = P.evaluate(lambda c, p: c["sim"], "CLIP", "test", seeds)
    c_val = P.candidates("val", None)
    res, rows = {"CLIP": base}, [base]
    for label, fn in (("CLIP+presence (no relations)", presence_score), (name, scorer)):
        alpha, curve = tune_alpha(c_val, fn(c_val, P))  # alpha on VAL only
        r = P.evaluate(lambda c, p, fn=fn, a=alpha: fuse(c["sim"], fn(c, p), a), f"{label} (a={alpha})", "test", seeds)
        r["alpha"], r["alpha_curve"] = alpha, curve
        r["bootstrap_vs_clip"] = paired_bootstrap(base["full"]["ranks"], r["full"]["ranks"])
        res[label] = r
        rows.append(r)
    if adapter is not None:
        from adapter import adapted_sim
        asim = lambda c: adapted_sim(adapter, P, c)
        ad = P.evaluate(lambda c, p: asim(c), "CLIP+Adapter", "test", seeds)
        ad["bootstrap_vs_clip"] = paired_bootstrap(base["full"]["ranks"], ad["full"]["ranks"])
        res["CLIP+Adapter"] = ad
        rows.append(ad)
        c_val_a = dict(c_val, sim=asim(c_val))  # val với cosine của adapter: alpha chọn trên VAL
        for label, fn in (("CLIP+Adapter+presence (no relations)", presence_score), (f"CLIP+Adapter+{name.replace('CLIP+', '')}", scorer)):
            alpha, curve = tune_alpha(c_val_a, fn(c_val, P))
            r = P.evaluate(lambda c, p, fn=fn, a=alpha: fuse(asim(c), fn(c, p), a), f"{label} (a={alpha})", "test", seeds)
            r["alpha"], r["alpha_curve"] = alpha, curve
            r["bootstrap_vs_clip"] = paired_bootstrap(base["full"]["ranks"], r["full"]["ranks"])
            r["bootstrap_vs_adapter"] = paired_bootstrap(ad["full"]["ranks"], r["full"]["ranks"])
            res[label] = r
            rows.append(r)
    print(summary_table(rows, title="text->image, test, 1K galleries, mean +- std over folds"))
    for g in base["by_group"]:
        print(summary_table([dict(name=f"{r['name']} - {g}", split="test", mean_std=r["by_group"][g]) for r in rows],
                            title=f"query group: {g}"))
    for k, r in res.items():
        if k != "CLIP":
            print(k, "bootstrap vs CLIP (full test pool):", json.dumps(r["bootstrap_vs_clip"]))
        if "bootstrap_vs_adapter" in r:
            print(k, "bootstrap vs CLIP+Adapter (full test pool):", json.dumps(r["bootstrap_vs_adapter"]))
    return res


def _load_protocol(work):
    from mmxai_eval import Protocol
    return Protocol.load(work)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("mode", choices=["build", "eval"])
    ap.add_argument("--work", required=True, help="the WORK dir (data/, split/, features/)")
    ap.add_argument("--repo", required=True)
    ap.add_argument("--name", default="COCO_full")
    ap.add_argument("--variant", default="full", choices=list(SUFFIX))
    ap.add_argument("--n-train-images", type=int, default=6000)
    ap.add_argument("--ckpt", default=None)
    ap.add_argument("--dim", type=int, default=200)
    ap.add_argument("--margin", type=float, default=12.0)
    ap.add_argument("--seeds", type=int, nargs="+", default=[0, 1, 2])
    ap.add_argument("--out", default="results_protocol.json")
    ap.add_argument("--skip-nn", action="store_true")
    ap.add_argument("--adapter", action="store_true", help="eval: huấn luyện adapter (train/val) và thêm CLIP+Adapter(+AdaMF-MAT)")
    a = ap.parse_args()
    P = _load_protocol(a.work)
    if a.mode == "build":
        from features import Encoder
        enc = Encoder("clip")  # same OpenCLIP ViT-B-32 laion2b as the shared features
        build_mmkg(P, a.repo, a.name, enc.texts, a.variant, a.n_train_images, run_nn=not a.skip_nn)
    else:
        sc = AdamfScorer(P, a.repo, a.name, a.ckpt, a.dim, a.margin)
        adapter = None
        if a.adapter:
            from adapter import train_adapter
            adapter = train_adapter(P, seed=0)
        res = evaluate_all(P, sc, a.seeds, adapter=adapter)
        from mmxai_eval import save_json
        save_json(res, a.out)
        print("saved", a.out)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/src/adapter.py
"""Adapter nhẹ trên đặc trưng CLIP đóng băng — phần MỞ RỘNG, không thuộc AdaMF-MAT.

Cùng ý tưởng adapter (`src/mmxai/graph/qa.py`: q_proj / skip khởi tạo = ma trận đơn vị, InfoNCE, dừng sớm theo MRR val):
hai phép tuyến tính D×D (caption, ảnh) khởi tạo = đơn vị, huấn luyện InfoNCE (tau 0.05) trên caption + ảnh của split TRAIN, chọn epoch theo MRR trên VAL.
Test không được dùng. Điểm adapter thay cosine CLIP trong cùng top-K ứng viên của giao thức chung.

Vì sao có file này: adapter tự nó đã tăng MRR (QA-Graph: 60.40 → 63.14 toàn pool). Để biết AdaMF-MAT đóng góp bao nhiêu thì phải so
"CLIP+Adapter" với "CLIP+Adapter+AdaMF-MAT", không so với CLIP thô.
"""
import copy

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


class Adapter(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.q = nn.Linear(dim, dim, bias=False)
        self.i = nn.Linear(dim, dim, bias=False)
        nn.init.eye_(self.q.weight)
        nn.init.eye_(self.i.weight)

    def text(self, x):
        return F.normalize(self.q(x), dim=-1)

    def image(self, x):
        return F.normalize(self.i(x), dim=-1)


@torch.no_grad()
def adapted_sim(model, P, c, device="cpu", chunk=500):
    """cos(adapter(caption), adapter(ảnh ứng viên)) → [Nq, K], cùng hình dạng với c["sim"]."""
    model.eval().to(device)
    out = np.zeros(c["sim"].shape, np.float32)
    for s in range(0, len(out), chunk):
        q = torch.from_numpy(P.cap[c["q_cap"][s:s + chunk]]).to(device)
        x = torch.from_numpy(P.img[c["cand"][s:s + chunk]]).to(device)  # [b, K, D]
        out[s:s + chunk] = torch.einsum("bd,bkd->bk", model.text(q), model.image(x)).cpu().numpy()
    return out


def _val_mrr(model, P, device):
    from mmxai_eval import rank_metrics, rerank_ranks
    c = P.candidates("val", None)
    return rank_metrics(rerank_ranks(c, adapted_sim(model, P, c, device)))["MRR"]


def train_adapter(P, seed=0, epochs=20, patience=4, batch_size=512, tau=0.05, lr=1e-3, weight_decay=1e-4, device=None, log=print):
    """Huấn luyện trên caption/ảnh TRAIN; trả về mô hình ở epoch có MRR val tốt nhất."""
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    torch.manual_seed(seed)
    model = Adapter(P.img.shape[1]).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    train_imgs = P.pool("train")
    cap_idx = np.array([ci for ii in train_imgs for ci in P.caps_of.get(ii, [])])
    cap_img = P.cap_img[cap_idx]
    Q = torch.from_numpy(P.cap[cap_idx]).to(device)
    X = torch.from_numpy(P.img).to(device)
    R = torch.from_numpy(cap_img.astype(np.int64)).to(device)
    best, best_state, bad, hist = -1.0, None, 0, []
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(len(Q), device=device)
        tot = 0.0
        for s in range(0, len(perm), batch_size):
            b = perm[s:s + batch_size]
            imgs = R[b]
            logits = model.text(Q[b]) @ model.image(X[imgs]).T / tau
            same = (imgs[:, None] == imgs[None, :]) & ~torch.eye(len(b), dtype=torch.bool, device=device)  # cùng ảnh ≠ âm
            loss = F.cross_entropy(logits.masked_fill(same, -1e9), torch.arange(len(b), device=device))
            opt.zero_grad()
            loss.backward()
            opt.step()
            tot += float(loss.detach()) * len(b)
        v = _val_mrr(model, P, device)
        hist.append({"epoch": ep, "loss": tot / len(perm), "val_MRR": v})
        log(f"  adapter epoch {ep:2d} loss {tot / len(perm):.4f} val_MRR {v:.2f}")
        if v > best + 1e-4:
            best, best_state, bad = v, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= patience:
                break
    model.load_state_dict(best_state)
    model.history, model.best_val_mrr = hist, best
    return model.eval()


In [ ]:
@stage("3 build the MMKG variants from the protocol (CLIP text features for concept names)")
def _():
    if REUSE_RUN: log("skipped (REUSE_RUN)"); return
    if not STATE["work"]: STATE["ok"] = False; log("skipped (no WORK)"); return
    for v in VARIANTS:
        rc, o = sh(f"python adamf_protocol.py build --work {STATE['work']} --repo {R} --name {NAMES[v]} --variant {v} "
                   f"--n-train-images {N_TRAIN_IMAGES}", cwd=SRC, timeout=3600)
        log(f"[{v}] rc={rc}\n{o[-1500:]}")
        if rc != 0: STATE["ok"] = False; return

In [ ]:
def train(name, epochs, tag, gpu=0, timeout_s=None, wait=True):
    os.makedirs(f"{R}/checkpoint", exist_ok=True)
    logf = f"{OUT}/{name}-{tag}.log"
    tmo = f"timeout {timeout_s} " if timeout_s else ""
    cmd = (f"CUDA_VISIBLE_DEVICES={gpu} {tmo}python run_adamf_mat.py -dataset={name} -batch_size=1024 -margin=12 "
           f"-epoch={epochs} -dim=200 -mu=0 -save=./checkpoint/{name}-{tag} -neg_num=128 -learning_rate={LR} > {logf} 2>&1")
    t = time.time(); p = subprocess.Popen(cmd, shell=True, cwd=R)
    if not wait: return p, logf, t
    rc = p.wait()
    log(f"{name} {tag}: rc={rc} time={time.time()-t:.0f}s\n--- log tail ---\n{open(logf).read()[-1500:]}")
    return rc, logf

def finish(name, p, logf, t):
    rc = p.wait()
    log(f"{name} full: rc={rc} time={time.time()-t:.0f}s\n--- log tail ---\n{open(logf).read()[-1500:]}")
    if rc != 0: STATE["ok"] = False

@stage("4 speed test: 2 epochs on the full variant (also checks the repo accepts the files)")
def _():
    if REUSE_RUN: log("skipped (REUSE_RUN)"); return
    if not STATE["ok"]: log("skipped (earlier failure)"); return
    rc, logf = train(NAMES["full"], 2, "smoke", timeout_s=3600)
    if rc != 0: STATE["ok"] = False; return
    sec = float(re.findall(r"([\d.]+)s/it", open(logf).read())[-1])
    STATE["epochs"] = max(2, min(EPOCH_CAP, int(BUDGET_H * 3600 / sec)))
    log(f"{sec:.1f} s/epoch -> {STATE['epochs']} epochs for EVERY variant (budget {BUDGET_H} h per phase)")

In [ ]:
@stage("5 train AdaMF-MAT: phase A = full (GPU0) + shuffled edges (GPU1) in parallel; phase B = dropped edges")
def _():
    if REUSE_RUN or not STATE["ok"] or not STATE["epochs"]: log("skipped"); return
    to = int((BUDGET_H + 1) * 3600); ep = STATE["epochs"]
    phaseA = [("full", 0)] + ([("shuffle", 1 if N_GPU > 1 else 0)] if "shuffle" in VARIANTS else [])
    ps = [(NAMES[v], *train(NAMES[v], ep, "full", g, to, wait=False)) for v, g in phaseA]
    if N_GPU <= 1:  # one GPU: run them one after another
        for n, p, lf, t in ps: finish(n, p, lf, t)
    else:
        for n, p, lf, t in ps: finish(n, p, lf, t)
    if "drop" in VARIANTS and STATE["ok"]:
        p, lf, t = train(NAMES["drop"], ep, "full", 0, to, wait=False); finish(NAMES["drop"], p, lf, t)

In [ ]:
@stage("6 evaluate each variant with the team protocol (3 folds + full pool, alpha tuned on val)")
def _():
    if not STATE["ok"]: log("skipped"); return
    for v in VARIANTS:
        n = NAMES[v]
        rc, o = sh(f"python adamf_protocol.py eval --work {STATE['work']} --repo {R} --name {n} --ckpt checkpoint/{n}-full "
                   f"--seeds {' '.join(map(str, SEEDS))} {'--adapter' if USE_ADAPTER else ''} --out {OUT}/results_{n}.json", cwd=SRC, timeout=7200)
        log(f"== {n}: rc={rc}\n{o[-3500:]}")

In [ ]:
@stage("7 summary")
def _():
    for v in VARIANTS:
        n = NAMES[v]; f = f"{OUT}/results_{n}.json"
        if not os.path.exists(f): log(n, ": no results"); continue
        r = json.load(open(f))
        log(f"[{n}]  test, 1K galleries, mean over folds (R@1 / R@10 / MRR) and by query group (MRR)")
        for k, x in r.items():
            ms = x["mean_std"]
            grp = {g: round(x["by_group"][g]["MRR"]["mean"], 2) for g in x.get("by_group", {})}
            log(f"   {k:<34} R@1 {ms['R@1']['mean']:.2f}  R@10 {ms['R@10']['mean']:.2f}  MRR {ms['MRR']['mean']:.2f}  {grp}")
    for v in VARIANTS:
        lf = f"{OUT}/{NAMES[v]}-full.log"
        if os.path.exists(lf):
            avg = [l.strip() for l in open(lf, errors="ignore") if "averaged(filter)" in l]
            log(f"KGC on held-out concept relations [{NAMES[v]}]:", avg[-1] if avg else "n/a")
log("\nDONE. Send me /kaggle/working/report_protocol.txt and results_PSG_*.json (small files).")